# MLP Model — pIC50 Prediction (ChEMBL)
Scaffold: basic MLP with dying-neuron checks, dropout, gradient monitoring, and R² tracking.

In [ ]:
# --- 0. Dependencies ---
import subprocess, sys
for pkg in ["rdkit-pypi", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("OK")

In [ ]:
# --- 1. Imports ---
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from rdkit import Chem
from rdkit.Chem import Descriptors, QED as QED_module
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# --- 2. Config ---
SILVER_DIR    = os.path.join("chembl_work", "parquet_silver")
TARGETS_PATH  = "targets_silver.parquet"   # in repo root (tid, organism only)
MAX_SAMPLES   = 10_000   # cap per protein (as per project spec)
BATCH_SIZE    = 256
LR            = 1e-3
NUM_EPOCHS    = 100
HIDDEN_DIMS   = [256, 128, 64]
DROPOUT       = 0.3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 15       # early stopping

In [ ]:
# --- 3. Load silver layer ---
print("Loading silver parquets...")
df_act = pd.read_parquet(os.path.join(SILVER_DIR, "activities_silver.parquet"))
df_ass = pd.read_parquet(os.path.join(SILVER_DIR, "assays_silver.parquet"))
df_mol = pd.read_parquet(os.path.join(SILVER_DIR, "molecules_silver.parquet"))

# Silver targets: only tid + organism (no pref_name)
df_tgt = pd.read_parquet(TARGETS_PATH)
df_tgt = df_tgt[df_tgt["organism"] == "Homo sapiens"].copy()
df_tgt["tid"] = df_tgt["tid"].astype(int)

print(f"  Activities:  {df_act.shape}")
print(f"  Assays:      {df_ass.shape}")
print(f"  Molecules:   {df_mol.shape}")
print(f"  Targets (HS):{df_tgt.shape}")

In [ ]:
# --- 4. Join and find top Homo sapiens protein ---
df = df_act.merge(df_ass, on="assay_id", how="inner")
df = df.merge(df_tgt[["tid"]], on="tid", how="inner")   # inner -> HS only
df = df.merge(df_mol, on="molregno", how="inner")
df = df.dropna(subset=["canonical_smiles", "pIC50"])
df = df[df["canonical_smiles"].str.len() > 0]

# Top protein by number of IC50 measurements (tid only — no pref_name in silver)
top_protein_counts = df.groupby("tid").size().sort_values(ascending=False)
print("Top 10 Homo sapiens proteins by IC50 count (by TID):")
print(top_protein_counts.head(10).to_string())

# Pick the top one
TOP_TID  = int(top_protein_counts.index[0])
TOP_NAME = f"TID_{TOP_TID}"
print(f"\nSelected protein: TID={TOP_TID}")

In [ ]:
# --- 5. Filter to one protein + cap at MAX_SAMPLES ---
df_protein = df[df["tid"] == TOP_TID].copy()
print(f"Total samples for {TOP_NAME}: {len(df_protein):,}")

# pIC50 sanity filter: keep [2, 12]
df_protein = df_protein[(df_protein["pIC50"] >= 2) & (df_protein["pIC50"] <= 12)]

# Sample if too large
if len(df_protein) > MAX_SAMPLES:
    df_protein = df_protein.sample(n=MAX_SAMPLES, random_state=SEED)
    print(f"Sampled to {MAX_SAMPLES:,}")
else:
    print(f"Using all {len(df_protein):,} samples (< {MAX_SAMPLES:,})")

df_protein = df_protein.reset_index(drop=True)
print(f"pIC50 range: [{df_protein['pIC50'].min():.2f}, {df_protein['pIC50'].max():.2f}]")

In [ ]:
# --- 6. RDKit descriptors ---
DESC_COLS = ["MW", "LogP", "HBD", "HBA", "TPSA", "QED", "RotBonds", "AromaticRings"]

def compute_descriptors(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return [np.nan] * len(DESC_COLS)
        return [
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol),
            QED_module.qed(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol),
        ]
    except Exception:
        return [np.nan] * len(DESC_COLS)

print(f"Computing RDKit descriptors for {len(df_protein):,} molecules...")
results = [compute_descriptors(s) for s in df_protein["canonical_smiles"]]
desc_df = pd.DataFrame(results, columns=DESC_COLS)
df_protein = pd.concat([df_protein.reset_index(drop=True), desc_df], axis=1)
df_protein = df_protein.dropna(subset=DESC_COLS)
print(f"After dropping failed SMILES: {len(df_protein):,}")
print(df_protein[DESC_COLS].describe().round(2))

In [ ]:
# --- 7. Train / Val / Test split + StandardScaler ---
FEATURES = DESC_COLS   # 8 molecular descriptors
TARGET   = "pIC50"

X = df_protein[FEATURES].values.astype(np.float32)
y = df_protein[TARGET].values.astype(np.float32)

# 70 / 15 / 15
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val,   X_test, y_val,   y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# Scale features (fit on train only)
feat_scaler = StandardScaler()
X_train = feat_scaler.fit_transform(X_train)
X_val   = feat_scaler.transform(X_val)
X_test  = feat_scaler.transform(X_test)

# Scale target (fit on train only)
target_scaler = StandardScaler()
y_train_s = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s   = target_scaler.transform(y_val.reshape(-1, 1)).ravel()

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"y range  train=[{y_train.min():.2f}, {y_train.max():.2f}]  "
      f"val=[{y_val.min():.2f}, {y_val.max():.2f}]")

In [ ]:
# --- 8. PyTorch Datasets ---
def to_tensors(*arrays):
    return [torch.tensor(a, dtype=torch.float32).to(DEVICE) for a in arrays]

Xt, yt   = to_tensors(X_train, y_train_s)
Xv, yv   = to_tensors(X_val,   y_val_s)
Xte, yte = to_tensors(X_test,  y_test)   # test y is UN-scaled for final R²

train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=BATCH_SIZE)
print(f"Batches/epoch: {len(train_loader)}")

In [ ]:
# --- 9. MLP Model ---
class MLP(nn.Module):
    """
    Multi-Layer Perceptron for pIC50 regression.

    Design choices:
    - LeakyReLU (slope=0.01): prevents dying neurons (dead ReLU problem)
    - BatchNorm before activation: stabilises training
    - Dropout: regularisation
    - Kaiming (He) init for weights: appropriate for LeakyReLU
    """
    def __init__(self, input_dim: int, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT):
        super().__init__()
        layers = []
        in_d = input_dim
        for i, h in enumerate(hidden_dims):
            layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.LeakyReLU(negative_slope=0.01),
                nn.Dropout(dropout if i < len(hidden_dims) - 1 else dropout * 0.5),
            ]
            in_d = h
        layers.append(nn.Linear(in_d, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, a=0.01, nonlinearity="leaky_relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLP(input_dim=len(FEATURES)).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# --- 10. Training utilities ---

def gradient_norm(model) -> float:
    """L2 norm of all gradients (scalar measure of gradient magnitude)."""
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5


def dead_neuron_pct(model, X_sample: torch.Tensor) -> dict:
    """
    Forward-pass through model with hooks to measure % dead activations.
    A neuron is 'dead' if its LeakyReLU output <= 0 (technically it still
    has a small gradient, but the check flags near-zero activations).
    """
    stats = {}
    hooks = []

    def make_hook(name):
        def _hook(module, inp, out):
            with torch.no_grad():
                # for LeakyReLU dead means output almost 0 (passed through negative)
                dead = (out <= 0).float().mean().item()
                stats[name] = dead
        return _hook

    for name, module in model.named_modules():
        if isinstance(module, nn.LeakyReLU):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        _ = model(X_sample)
    model.train()
    for h in hooks:
        h.remove()
    return stats


def r2(y_true_np, y_pred_np):
    return r2_score(y_true_np, y_pred_np)

In [ ]:
# --- 11. Training loop ---
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)

history = {"train_loss": [], "val_loss": [], "val_r2": [], "grad_norm": [], "grad_delta": []}
best_val_loss = float("inf")
best_state    = None
patience_ctr  = 0
prev_grad_norm = 0.0

# Small sample for dead-neuron check (first 256 train samples)
X_probe = Xt[:256]

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- train ----
    model.train()
    train_loss = 0.0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        # Gradient clipping (prevents exploding gradients)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_loss += loss.item() * len(Xb)
    train_loss /= len(Xt)

    # ---- gradient stats ----
    g_norm  = gradient_norm(model)
    g_delta = abs(g_norm - prev_grad_norm)
    prev_grad_norm = g_norm

    # ---- validation ----
    model.eval()
    with torch.no_grad():
        val_pred_s = model(Xv).cpu().numpy()
        val_loss   = criterion(model(Xv), yv).item()
        # Inverse-scale to original pIC50 for R²
        val_pred   = target_scaler.inverse_transform(val_pred_s.reshape(-1, 1)).ravel()
        val_r2_val = r2(y_val, val_pred)

    scheduler.step(val_loss)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_r2"].append(val_r2_val)
    history["grad_norm"].append(g_norm)
    history["grad_delta"].append(g_delta)

    # ---- early stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stop at epoch {epoch}")
            break

    if epoch % 10 == 0 or epoch == 1:
        dead = dead_neuron_pct(model, X_probe)
        dead_str = "  ".join(f"{k}: {v*100:.1f}%" for k, v in dead.items())
        print(f"Ep {epoch:03d} | train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_R²={val_r2_val:.4f}  |  grad_norm={g_norm:.5f}  Δgrad={g_delta:.5f}")
        print(f"         dead activations -> {dead_str}")

print("\nLoading best model weights...")
model.load_state_dict(best_state)

In [ ]:
# --- 12. Final evaluation on test set ---
model.eval()
with torch.no_grad():
    test_pred_s = model(Xte).cpu().numpy()

test_pred = target_scaler.inverse_transform(test_pred_s.reshape(-1, 1)).ravel()
test_true = y_test  # already un-scaled

mse  = mean_squared_error(test_true, test_pred)
rmse = mse ** 0.5
r2v  = r2_score(test_true, test_pred)

print("=" * 45)
print(f"TEST SET RESULTS  ({len(test_true):,} samples)")
print("=" * 45)
print(f"  MSE  : {mse:.4f}")
print(f"  RMSE : {rmse:.4f}")
print(f"  R²   : {r2v:.4f}")
print("=" * 45)

In [ ]:
# --- 13. Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
ax = axes[0]
ax.plot(history["train_loss"], label="Train loss")
ax.plot(history["val_loss"],   label="Val loss")
ax.set_title("Loss (MSE, normalised pIC50)")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE"); ax.legend()

# R² curve
ax = axes[1]
ax.plot(history["val_r2"], color="green", label="Val R²")
ax.axhline(r2v, color="red", linestyle="--", label=f"Test R²={r2v:.3f}")
ax.set_title("Validation R²"); ax.set_xlabel("Epoch"); ax.legend()

# Gradient norm
ax = axes[2]
ax.plot(history["grad_norm"],  label="||grad||", color="purple")
ax.plot(history["grad_delta"], label="Δ||grad||", color="orange", alpha=0.7)
ax.set_title("Gradient norm per epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("L2 norm"); ax.legend()

plt.tight_layout(); plt.show()

# Actual vs Predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(test_true, test_pred, alpha=0.4, s=15, label="Predictions")
mn, mx = min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())
plt.plot([mn, mx], [mn, mx], "r--", label="Perfect fit")
plt.xlabel("True pIC50"); plt.ylabel("Predicted pIC50")
plt.title(f"Actual vs Predicted  (R²={r2v:.3f})")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- 14. Overfitting check on tiny subset ---
TINY_N = 200
print(f"=== Overfitting check: training on only {TINY_N} samples ===")

X_tiny, _, y_tiny, _ = train_test_split(X_train, y_train_s, train_size=TINY_N, random_state=SEED)
Xt2, yt2 = to_tensors(X_tiny, y_tiny)
tiny_loader = DataLoader(TensorDataset(Xt2, yt2), batch_size=32, shuffle=True)

tiny_model = MLP(input_dim=len(FEATURES)).to(DEVICE)
tiny_opt   = torch.optim.Adam(tiny_model.parameters(), lr=LR)
tiny_crit  = nn.MSELoss()

tiny_train_losses, tiny_val_losses = [], []
for epoch in range(1, 151):
    tiny_model.train()
    for Xb, yb in tiny_loader:
        tiny_opt.zero_grad()
        tiny_crit(tiny_model(Xb), yb).backward()
        tiny_opt.step()
    with torch.no_grad():
        tiny_model.eval()
        tl = tiny_crit(tiny_model(Xt2), yt2).item()
        vl = tiny_crit(tiny_model(Xv), yv).item()
        tiny_train_losses.append(tl)
        tiny_val_losses.append(vl)

plt.figure(figsize=(8, 4))
plt.plot(tiny_train_losses, label=f"Train loss (n={TINY_N})")
plt.plot(tiny_val_losses,   label="Val loss (full val set)")
plt.title(f"Overfitting check — n={TINY_N}")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend()
plt.tight_layout(); plt.show()

train_min = min(tiny_train_losses)
val_min   = min(tiny_val_losses)
gap = val_min - train_min
print(f"Min train loss: {train_min:.4f}  |  Min val loss: {val_min:.4f}  |  Gap: {gap:.4f}")
if gap > 0.3:
    print("WARNING: model is likely overfitting — val >> train. Increase dropout or reduce capacity.")
else:
    print("Gap looks reasonable.")